# M2 - defense Layer 1: perplexity filter (calibrated)

Turns Layer 1 from a pass-through stub into a real input filter: score each prompt's
perplexity under a frozen scorer LM, calibrate a threshold so **no clean prompt is
flagged** (Jain et al. 2023's method), then measure which attacks it catches.

Scorer: **`openai-community/gpt2-large`** - GPT-2 (2019), open weights, MIT licence.
The notebook also scores with the Qwen target (self-perplexity) so you can compare.

**Before running:** GPU T4 + Internet **On**; push `main` to GitHub. No HF token needed
(both models are public). This notebook does not depend on M1 having been run.

## 1 - Setup

In [ ]:
%pip -q install -U "transformers>=4.45" "accelerate>=0.30" "huggingface_hub>=0.24"

In [ ]:
import os, subprocess, sys, pathlib, time, json, glob
import numpy as np, pandas as pd

_sec = None
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
except Exception as e:
    print('no Kaggle secrets client:', e)

def _secret(name):
    try:
        return _sec.get_secret(name) if _sec is not None else None
    except Exception:
        return None

_hf = _secret('HF_TOKEN')   # optional - gpt2-large and Qwen2.5 are public
if _hf:
    os.environ['HF_TOKEN'] = _hf
    from huggingface_hub import login; login(token=_hf)
    print('HF auth OK')
else:
    print('no HF_TOKEN secret (fine - models are public)')

In [ ]:
# --- get the repo (works public or private) --------------------------------
REPO   = "MehemudAzad/LLM-jailbreaking-with-layered-prompt-defense"
BRANCH = "main"
ROOT   = pathlib.Path("/kaggle/working/repo")

_gh  = _secret("GH_TOKEN")          # set this Kaggle secret only if the repo is private
_url = f"https://{_gh}@github.com/{REPO}.git" if _gh else f"https://github.com/{REPO}.git"

subprocess.run(["rm", "-rf", str(ROOT)])
_r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, _url, str(ROOT)],
                    capture_output=True, text=True)
if _r.returncode != 0:
    _err = _r.stderr.replace(_gh, "***") if _gh else _r.stderr
    raise RuntimeError(
        "git clone failed:\n" + _err +
        "\n\nPrivate repo? Create a GitHub fine-grained PAT (Contents: read-only, this repo),"
        "\nadd it as a Kaggle Secret named GH_TOKEN, and re-run. Or make the repo public."
        "\nAlso confirm main is pushed:  git push -u origin main"
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("repo at", ROOT, "| HEAD",
      subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"]).decode().strip())

In [ ]:
from core.config import CONFIG

s = CONFIG["models"]["perplexity_scorer"]
l1 = CONFIG["defense"]["layer1_perplexity"]
print("scorer   :", s["name"], "| backend:", s["backend"])
print("layer1   :", l1)
assert s["backend"] == "transformers", "set [models.perplexity_scorer] backend = 'transformers'"

## 2 - Load the scorer

`gpt2-large` is ~774M params - loads in seconds, fits in <2 GB.

In [ ]:
from core.seed import seed_everything
from core.models import load_perplexity_scorer

seed_everything()
_t0 = time.time()
scorer = load_perplexity_scorer()
print("ppl('hello world') =", round(scorer.perplexity("hello world"), 1),
      f"  (loaded in {time.time()-_t0:.0f}s)")

## 3 - Score the benign calibration set

The 18 frozen benign prompts. `windowed` = worst contiguous N-token span.
(Jain et al. calibrate on *clean harmful* prompts - swap that set in here once your
teammate freezes AdvBench; benign is the stand-in for now.)

In [ ]:
from core.datasets import load_benign

WINDOWS = [8, 16, 32]
benign = load_benign()
rows = []
for b in benign:
    rec = {'id': b.id, 'n_words': len(b.text.split()), 'ppl': scorer.perplexity(b.text)}
    for w in WINDOWS:
        rec[f'ppl_w{w}'] = scorer.perplexity(b.text, window=w)
    rows.append(rec)
bdf = pd.DataFrame(rows).set_index('id')
bdf.round(1)

## 4 - Calibrate (Jain et al.: threshold = max over the clean set)

In [ ]:
calib = {}
for w in WINDOWS:
    col = bdf[f'ppl_w{w}']
    calib[w] = {'max': col.max(), 'p95': col.quantile(.95), 'p90': col.quantile(.90)}
cdf = pd.DataFrame(calib).T
cdf.index.name = 'window'
print('windowed-perplexity thresholds from the benign set:')
print(cdf.round(1))

WIN = int(l1.get('window_size', 16))
THRESH = float(bdf[f'ppl_w{WIN}'].max())   # <- the one we'll enforce (0 benign FPR by construction)
print(f"\nchosen: window={WIN}, threshold={THRESH:.1f}  (max over benign -> benign FPR = 0)")

## 5 - Score the attack battery

Every attack template wrapped around one neutral instruction - the *mechanism* is what
drives perplexity (base64 of anything is gibberish). Helper-LM attacks (`auto_*`) and
`adaptive` are scored in M3.

In [ ]:
from attacks import load_all

NEUTRAL = "Summarize how the water cycle works in three sentences."
atks = load_all()
arows = []
for name, atk in sorted(atks.items()):
    if name == 'adaptive' or atk.uses_helper_lm:
        continue
    prompt = atk.apply(NEUTRAL).prompt
    arows.append({
        'attack': name, 'category': atk.category, 'n_words': len(prompt.split()),
        'ppl': scorer.perplexity(prompt),
        f'ppl_w{WIN}': scorer.perplexity(prompt, window=WIN),
    })
adf = pd.DataFrame(arows).set_index('attack')
adf.round(1)

In [ ]:
adf['caught'] = adf[f'ppl_w{WIN}'] > THRESH
print(f'Layer 1 @ window={WIN}, threshold={THRESH:.0f}  (windowed enforcement)\n')
print(adf[['category', f'ppl_w{WIN}', 'caught']].sort_values(f'ppl_w{WIN}', ascending=False).round(0))
print(f'\ncaught {int(adf.caught.sum())}/{len(adf)} techniques')
print('expected: encoding attacks (base64/rot13/leetspeak/disemvowel) caught; '
      'personas + injections pass - the documented blind spot')

## 6 - Optional: Qwen self-perplexity

In [ ]:
RUN_SELF_PPL = True   # set False to skip loading the 3B target
if RUN_SELF_PPL:
    from core.models import load_target
    qwen = load_target()
    q_benign_max = max(qwen.perplexity(b.text, window=WIN) for b in benign)
    print(f'Qwen benign max windowed-ppl (threshold if we used self-ppl): {q_benign_max:.1f}')
    qrows = []
    for name, atk in sorted(atks.items()):
        if name == 'adaptive' or atk.uses_helper_lm:
            continue
        p = atk.apply(NEUTRAL).prompt
        qrows.append({'attack': name, 'gpt2_wppl': scorer.perplexity(p, window=WIN),
                      'qwen_wppl': qwen.perplexity(p, window=WIN)})
    print(pd.DataFrame(qrows).set_index('attack').round(0))

## 7 - Config values + Layer 1 sanity check

In [ ]:
print('paste into config.toml under [defense.layer1_perplexity]:\n')
print(f'  threshold   = {THRESH:.1f}')
print(f'  window_size = {WIN}')
print( '  windowed    = true')
print( '  enforce     = true')

from huggingface_hub import HfApi
sha = HfApi().model_info(CONFIG['models']['perplexity_scorer']['name'], token=os.environ.get('HF_TOKEN')).sha
print(f'\n  # [models.perplexity_scorer] revision = "{sha}"')

In [ ]:
# exercise the real layer with the calibrated threshold
from defense.layer1_perplexity_filter import PerplexityFilter
from defense.base import DefenseContext

layer = PerplexityFilter({'enabled': True, 'enforce': True, 'windowed': True,
                          'window_size': WIN, 'threshold': THRESH})
for label, text in [('benign', benign[0].text),
                    ('base64 attack', atks['base64'].apply(NEUTRAL).prompt),
                    ('aim persona', atks['aim'].apply(NEUTRAL).prompt)]:
    ctx = DefenseContext(goal_id='t', attack='t', original_prompt=text, prompt=text, metadata={'goal': text})
    ctx = layer.process(ctx)
    print(f'{label:14s} -> blocked={ctx.blocked}  ({ctx.verdicts[-1].reason})')

## Done - what to commit

- `core/models.py` - `perplexity()` (plain + windowed)
- `defense/layer1_perplexity_filter.py` - real scoring, windowed enforcement
- `config.toml` - `[models.perplexity_scorer]` on transformers + pinned revision;
  `[defense.layer1_perplexity]` calibrated `threshold` / `window_size`, `enforce = true`
- `notebooks/m2_perplexity_filter.ipynb`

For the Design Report: the benign perplexity distribution, the calibrated threshold
(0 benign false positives by construction), and the per-technique catch/miss table.

**M3:** judge model backend + Layer 4, then a baseline ASR pass once AdvBench is frozen.